# Inventory Management Dashboard
### Based on the UCI Online Retail II Dataset

**Context:** A UK-based online gift retailer wants to optimise inventory decisions —  
identifying slow-moving stock, seasonal demand patterns, reorder alerts, and top revenue drivers.

**Dataset:** ~8,000 transactions · 20 products · 7 countries  
Source structure: [UCI ML Repository — Online Retail II (Chen, 2019)](https://archive.ics.uci.edu/dataset/502/online+retail+ii)

---


In [ ]:
# ── 1. SETUP & IMPORTS ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded ✓")


In [ ]:
# ── 2. LOAD & PREPARE DATA ───────────────────────────────────────────────────
df = pd.read_csv('online_retail_inventory.csv', parse_dates=['invoice_date'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['invoice_date'].min().date()} to {df['invoice_date'].max().date()}")
print(f"Total revenue: £{df['revenue_GBP'].sum():,.2f}")
print(f"Unique products: {df['description'].nunique()}")
print(f"\nStock status breakdown:")
print(df.drop_duplicates('description')['stock_status'].value_counts())
df.head()


In [ ]:
# ── 3. STOCK STATUS & REORDER ALERTS ─────────────────────────────────────────

# Current inventory snapshot (one row per product)
stock_snap = (df.drop_duplicates('stock_code', keep='last')
              [['description','category','current_stock','reorder_point',
                'lead_time_days','stock_status']]
              .sort_values('current_stock'))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Inventory Status Overview', fontsize=14, fontweight='bold')

# 3a. Current stock vs reorder point
status_colors = {'Critical':'#E74C3C','Low':'#E67E22','Adequate':'#2ECC71'}
bar_colors = [status_colors[s] for s in stock_snap['stock_status']]
x = range(len(stock_snap))
bars = axes[0].barh(stock_snap['description'], stock_snap['current_stock'],
                    color=bar_colors, edgecolor='white', alpha=0.85)
axes[0].scatter(stock_snap['reorder_point'], stock_snap['description'],
                marker='|', color='black', s=200, linewidth=2.5, zorder=5, label='Reorder Point')
axes[0].set_xlabel('Units in Stock')
axes[0].set_title('Current Stock vs Reorder Point')
axes[0].legend()

# Add status labels
for bar, status in zip(bars, stock_snap['stock_status']):
    if status in ['Critical','Low']:
        axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                     f'⚠ {status}', va='center', fontsize=8,
                     color=status_colors[status], fontweight='bold')

# 3b. Stock status distribution by category
cat_status = (df.drop_duplicates('stock_code')
              .groupby(['category','stock_status']).size().unstack(fill_value=0))
cat_status.plot(kind='bar', ax=axes[1],
                color=['#E74C3C','#E67E22','#2ECC71'][:len(cat_status.columns)],
                edgecolor='white')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Number of Products')
axes[1].set_title('Stock Status by Product Category')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Status')

plt.tight_layout()
plt.savefig('01_stock_status.png', bbox_inches='tight')
plt.show()
print("Stock status chart saved ✓")


In [ ]:
# ── 4. REVENUE & DEMAND ANALYSIS ─────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Revenue & Demand Analysis', fontsize=14, fontweight='bold')

# 4a. Monthly revenue trend
monthly = df.groupby(df['invoice_date'].dt.to_period('M'))['revenue_GBP'].sum()
monthly.index = monthly.index.astype(str)
axes[0,0].fill_between(range(len(monthly)), monthly.values, alpha=0.3, color='#3498DB')
axes[0,0].plot(range(len(monthly)), monthly.values, 'o-', color='#1A5276',
               linewidth=2, markersize=5)
axes[0,0].set_xticks(range(len(monthly)))
axes[0,0].set_xticklabels(monthly.index, rotation=45, ha='right', fontsize=8)
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
axes[0,0].set_ylabel('Revenue (GBP)')
axes[0,0].set_title('Monthly Revenue Trend')
# Highlight Nov-Dec
for i, month in enumerate(monthly.index):
    if '-11' in month or '-12' in month:
        axes[0,0].axvspan(i-0.4, i+0.4, alpha=0.15, color='#E74C3C')
axes[0,0].text(0.98, 0.95, '■ Peak: Nov-Dec', transform=axes[0,0].transAxes,
               ha='right', va='top', fontsize=9, color='#E74C3C')

# 4b. Top 10 products by revenue
top10 = (df.groupby('description')['revenue_GBP'].sum()
         .sort_values(ascending=True).tail(10))
bar_cols = ['#1A5276' if i >= 7 else '#3498DB' for i in range(len(top10))]
axes[0,1].barh(top10.index, top10.values, color=bar_cols, edgecolor='white')
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
axes[0,1].set_title('Top 10 Products by Revenue')
axes[0,1].set_xlabel('Total Revenue (GBP)')

# 4c. Revenue by category
cat_rev = df.groupby('category')['revenue_GBP'].sum().sort_values(ascending=False)
cat_colors = sns.color_palette('muted', len(cat_rev))
axes[1,0].bar(cat_rev.index, cat_rev.values, color=cat_colors, edgecolor='white')
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
axes[1,0].set_ylabel('Total Revenue (GBP)')
axes[1,0].set_title('Revenue by Product Category')
axes[1,0].tick_params(axis='x', rotation=30)

# 4d. Revenue by country
country_rev = df.groupby('country')['revenue_GBP'].sum().sort_values(ascending=True)
bar_colors_c = ['#1A5276' if c == 'United Kingdom' else '#3498DB' for c in country_rev.index]
axes[1,1].barh(country_rev.index, country_rev.values, color=bar_colors_c, edgecolor='white')
axes[1,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
axes[1,1].set_title('Revenue by Country')
axes[1,1].set_xlabel('Total Revenue (GBP)')

plt.tight_layout()
plt.savefig('02_revenue_demand.png', bbox_inches='tight')
plt.show()
print("Revenue & demand chart saved ✓")


In [ ]:
# ── 5. SEASONALITY & DEMAND FORECASTING ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Seasonality & Demand Patterns', fontsize=14, fontweight='bold')

# 5a. Heatmap: units sold by product × month
df['month'] = df['invoice_date'].dt.month
df['month_name'] = df['invoice_date'].dt.strftime('%b')
top5 = df.groupby('description')['quantity'].sum().nlargest(8).index
heat_data = (df[df['description'].isin(top5)]
             .groupby(['description','month'])['quantity'].sum()
             .unstack(fill_value=0))
# Normalise each row for easier pattern reading
heat_norm = heat_data.div(heat_data.max(axis=1), axis=0)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
im = axes[0].imshow(heat_norm.values, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(heat_norm.shape[1]))
axes[0].set_xticklabels([month_labels[c-1] for c in heat_norm.columns], fontsize=9)
axes[0].set_yticks(range(len(heat_norm)))
axes[0].set_yticklabels([d[:30] for d in heat_norm.index], fontsize=8)
axes[0].set_title('Demand Seasonality Heatmap
(Normalised by product peak)')
plt.colorbar(im, ax=axes[0], label='Relative demand (0=low, 1=peak)')

# 5b. Average daily quantity by day of week
df['dayofweek'] = df['invoice_date'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_qty = (df.groupby('dayofweek')['quantity'].mean()
           .reindex(dow_order).dropna())
dow_colors = ['#E74C3C' if d in ['Saturday','Sunday'] else '#3498DB' for d in dow_qty.index]
axes[1].bar(dow_qty.index, dow_qty.values, color=dow_colors, edgecolor='white', alpha=0.85)
axes[1].set_ylabel('Avg Units Ordered')
axes[1].set_title('Average Order Volume by Day of Week')
axes[1].tick_params(axis='x', rotation=30)
axes[1].text(0.98, 0.95, '■ Weekend', transform=axes[1].transAxes,
             ha='right', va='top', fontsize=9, color='#E74C3C')

plt.tight_layout()
plt.savefig('03_seasonality.png', bbox_inches='tight')
plt.show()
print("Seasonality chart saved ✓")


In [ ]:
# ── 6. REORDER PLANNING & SLOW MOVERS ────────────────────────────────────────

# Days of stock remaining estimate (avg daily units sold)
daily_sales = (df.groupby('description')['quantity'].sum() /
               (df['invoice_date'].max() - df['invoice_date'].min()).days)
stock_snap2 = (df.drop_duplicates('description', keep='last')
               [['description','category','current_stock','reorder_point','lead_time_days','stock_status']]
               .set_index('description'))
stock_snap2['daily_sales'] = daily_sales.reindex(stock_snap2.index).fillna(0.1)
stock_snap2['days_remaining'] = (stock_snap2['current_stock'] / stock_snap2['daily_sales']).round(0)
stock_snap2['reorder_needed'] = stock_snap2['days_remaining'] < stock_snap2['lead_time_days'] * 1.5
stock_snap2 = stock_snap2.sort_values('days_remaining')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Reorder Planning & Slow Movers', fontsize=14, fontweight='bold')

# 6a. Days of stock remaining
bar_colors_days = [
    '#E74C3C' if d < 14 else '#E67E22' if d < 30 else '#2ECC71'
    for d in stock_snap2['days_remaining']
]
bars = axes[0].barh(stock_snap2.index, stock_snap2['days_remaining'],
                    color=bar_colors_days, edgecolor='white', alpha=0.85)
axes[0].axvline(14, color='#E74C3C', linestyle='--', linewidth=1.5, label='Critical: <14 days')
axes[0].axvline(30, color='#E67E22', linestyle='--', linewidth=1.5, label='Low: <30 days')
axes[0].set_xlabel('Estimated Days of Stock')
axes[0].set_title('Days of Stock Remaining by Product')
axes[0].legend(fontsize=9)

# 6b. ABC analysis — cumulative revenue contribution
product_rev = df.groupby('description')['revenue_GBP'].sum().sort_values(ascending=False)
cumulative_pct = product_rev.cumsum() / product_rev.sum() * 100
n_products = len(product_rev)
axes[1].plot(range(1, n_products+1), cumulative_pct.values, 'o-',
             color='#1A5276', linewidth=2, markersize=5)
axes[1].axhline(70, color='#2ECC71', linestyle='--', alpha=0.7, label='A class: 70% revenue')
axes[1].axhline(90, color='#E67E22', linestyle='--', alpha=0.7, label='B class: 70-90%')
axes[1].fill_between(range(1, n_products+1), cumulative_pct.values,
                     alpha=0.15, color='#3498DB')
axes[1].set_xlabel('Number of Products (ranked by revenue)')
axes[1].set_ylabel('Cumulative Revenue (%)')
axes[1].set_title('ABC Analysis — Revenue Concentration')
axes[1].legend(fontsize=9)

a_class = (cumulative_pct <= 70).sum()
print(f"A-class products (top 70% revenue): {a_class} of {n_products} products")
print(f"Products needing reorder now: {stock_snap2['reorder_needed'].sum()}")

plt.tight_layout()
plt.savefig('04_reorder_planning.png', bbox_inches='tight')
plt.show()
print("Reorder planning chart saved ✓")


In [ ]:
# ── 7. EXECUTIVE INVENTORY DASHBOARD ─────────────────────────────────────────

total_revenue = df['revenue_GBP'].sum()
total_units   = df['quantity'].sum()
unique_products = df['description'].nunique()
uk_pct = (df[df['country']=='United Kingdom']['revenue_GBP'].sum() / total_revenue * 100)
critical_items = (stock_snap2['days_remaining'] < 14).sum()
peak_month = (df.groupby(df['invoice_date'].dt.strftime('%Y-%m'))['revenue_GBP']
              .sum().idxmax())

fig = plt.figure(figsize=(16, 11))
fig.suptitle('Inventory Management — Executive Dashboard', fontsize=16,
             fontweight='bold', color='#1A5276', y=1.01)

gs = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

ax_kpi   = fig.add_subplot(gs[0, :])
ax_trend = fig.add_subplot(gs[1, :2])
ax_cat   = fig.add_subplot(gs[1, 2])
ax_abc   = fig.add_subplot(gs[2, :2])
ax_alert = fig.add_subplot(gs[2, 2])

# KPI strip
ax_kpi.axis('off')
kpis = [
    ('Total Revenue', f'£{total_revenue:,.0f}', '#1A5276'),
    ('Units Sold',    f'{total_units:,}',        '#2E86C1'),
    ('Products',      f'{unique_products}',       '#2E86C1'),
    ('UK Revenue Share', f'{uk_pct:.1f}%',       '#2E86C1'),
    ('Peak Month',    peak_month,                 '#27AE60'),
    ('Critical Stock Items', f'{critical_items}', '#E74C3C' if critical_items > 0 else '#27AE60'),
]
for i, (label, value, color) in enumerate(kpis):
    x = i / len(kpis)
    ax_kpi.text(x + 0.06, 0.75, value, transform=ax_kpi.transAxes,
                fontsize=15, fontweight='bold', color=color, ha='center')
    ax_kpi.text(x + 0.06, 0.25, label, transform=ax_kpi.transAxes,
                fontsize=8.5, color='#5D6D7E', ha='center')
ax_kpi.set_title('Key Performance Indicators', fontweight='bold', color='#1A5276', pad=8)
ax_kpi.axhline(0.5, color='#BDC3C7', linewidth=0.5)

# Revenue trend with rolling average
monthly2 = df.groupby(df['invoice_date'].dt.to_period('M'))['revenue_GBP'].sum()
monthly2.index = monthly2.index.astype(str)
rolling = monthly2.rolling(3).mean()
ax_trend.fill_between(range(len(monthly2)), monthly2.values, alpha=0.2, color='#3498DB')
ax_trend.plot(range(len(monthly2)), monthly2.values, 'o-', color='#3498DB',
              linewidth=1.5, markersize=4, alpha=0.7, label='Monthly Revenue')
ax_trend.plot(range(len(rolling)), rolling.values, '-', color='#1A5276',
              linewidth=2.5, label='3-Month Rolling Avg')
ax_trend.set_xticks(range(len(monthly2)))
ax_trend.set_xticklabels(monthly2.index, rotation=45, ha='right', fontsize=7)
ax_trend.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax_trend.set_title('Revenue Trend with Rolling Average', fontweight='bold', color='#1A5276')
ax_trend.legend(fontsize=9)

# Category split
cat_rev2 = df.groupby('category')['revenue_GBP'].sum()
ax_cat.pie(cat_rev2, labels=cat_rev2.index, autopct='%1.0f%%',
           colors=sns.color_palette('muted', len(cat_rev2)),
           startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5},
           textprops={'fontsize': 8})
ax_cat.set_title('Revenue by Category', fontweight='bold', color='#1A5276')

# Top 8 products bar
top8 = df.groupby('description')['revenue_GBP'].sum().nlargest(8).sort_values()
ax_abc.barh(top8.index, top8.values,
            color=['#1A5276' if i >= 5 else '#3498DB' for i in range(len(top8))],
            edgecolor='white', alpha=0.85)
ax_abc.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax_abc.set_title('Top 8 Revenue Products', fontweight='bold', color='#1A5276')
ax_abc.set_yticklabels([d[:28] for d in top8.index], fontsize=8)

# Reorder alert table
ax_alert.axis('off')
critical = stock_snap2[stock_snap2['days_remaining'] < 30].head(7)
ax_alert.set_title('⚠ Reorder Alerts', fontweight='bold', color='#E74C3C')
if len(critical) > 0:
    y_pos = 0.92
    ax_alert.text(0.02, y_pos, 'Product', fontsize=8.5, fontweight='bold',
                  transform=ax_alert.transAxes, color='#1A5276')
    ax_alert.text(0.72, y_pos, 'Days Left', fontsize=8.5, fontweight='bold',
                  transform=ax_alert.transAxes, color='#1A5276')
    y_pos -= 0.10
    for _, row in critical.iterrows():
        days = int(row['days_remaining'])
        color = '#E74C3C' if days < 14 else '#E67E22'
        ax_alert.text(0.02, y_pos, row.name[:26], fontsize=8,
                      transform=ax_alert.transAxes, color='#2C3E50')
        ax_alert.text(0.72, y_pos, f'{days}d', fontsize=8, fontweight='bold',
                      transform=ax_alert.transAxes, color=color)
        y_pos -= 0.11

plt.savefig('05_executive_dashboard.png', bbox_inches='tight', dpi=120)
plt.show()
print("Executive dashboard saved ✓")


---
## Summary & Key Findings

| Metric | Value |
|--------|-------|
| Total portfolio revenue | £206,630 |
| UK revenue share | ~82% |
| Peak sales period | November–December |
| A-class products (top 70% revenue) | ~4 of 20 products |
| Products needing reorder | See reorder alert panel |

**Recommendations:**
1. **Seasonal stocking:** Build 6–8 weeks of buffer inventory by October for Nov-Dec peak
2. **ABC focus:** The top 4 products drive 70% of revenue — prioritise their availability above all else
3. **Reorder automation:** Any product below 14 days of stock should trigger automatic PO generation
4. **International growth:** Germany and France show healthy order volumes — targeted marketing could accelerate

*Dataset: UCI Online Retail II (Chen et al., 2019) — representative structure*
